In [1]:
# ============================================
# CELL 1: Setup — Run this first!
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
# Create folder structure
base_path = '/content/drive/MyDrive/capstone_project'
for folder in ['raw_data', 'processed_data', 'notebooks', 'outputs']:
    os.makedirs(f'{base_path}/{folder}', exist_ok=True)

# Install required packages
!pip install sodapy openmeteo-requests requests-cache retry-requests geopandas shapely -q

print("✅ Setup complete!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.5/723.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 91.6 MB/s eta 0:00:00
✅ Setup complete!


In [3]:
# ============================================
# CELL 2: Download NYC Crime Data
# This downloads NYPD complaint data 2015-2020
# ============================================
import pandas as pd
from sodapy import Socrata

print("⏳ Downloading NYC crime data... This will take 5-10 minutes...")

# Connect to NYC Open Data API (no token needed for public data)
client = Socrata("data.cityofnewyork.us", None)

# Download in chunks — we need 2015 to 2020
all_data = []
years = [2015, 2016, 2017, 2018, 2019, 2020]

for year in years:
    print(f"  Downloading {year}...")
    results = client.get(
        "qgea-i56i",
        where=f"cmplnt_fr_dt >= '{year}-01-01T00:00:00' AND cmplnt_fr_dt < '{year+1}-01-01T00:00:00'",
        limit=500000,
        select="cmplnt_fr_dt, addr_pct_cd, ofns_desc, law_cat_cd, latitude, longitude"
    )
    df_year = pd.DataFrame.from_records(results)
    all_data.append(df_year)
    print(f"    → Got {len(df_year)} records for {year}")

crime_raw = pd.concat(all_data, ignore_index=True)
print(f"\n✅ Total records downloaded: {len(crime_raw)}")
print(crime_raw.head())

# Save raw data
crime_raw.to_csv(f'{base_path}/raw_data/nyc_crime_raw.csv', index=False)
print(f"✅ Saved to Google Drive: raw_data/nyc_crime_raw.csv")

⏳ Downloading NYC crime data... This will take 5-10 minutes...
    → Got 479100 records for 2015
    → Got 478711 records for 2016
    → Got 468470 records for 2017
    → Got 462836 records for 2018
    → Got 459481 records for 2019
    → Got 414074 records for 2020

✅ Total records downloaded: 2762672
              cmplnt_fr_dt addr_pct_cd                        ofns_desc  \
0  2015-01-01T00:00:00.000          40  MURDER & NON-NEGL. MANSLAUGHTER   
1  2015-01-01T00:00:00.000         122   CRIMINAL MISCHIEF & RELATED OF   
2  2015-01-01T00:00:00.000         123   OFFENSES AGAINST PUBLIC ADMINI   
3  2015-01-01T00:00:00.000         120     NYS LAWS-UNCLASSIFIED FELONY   
4  2015-01-01T00:00:00.000         122   CRIMINAL MISCHIEF & RELATED OF   

    law_cat_cd          latitude         longitude  
0       FELONY  40.8103518634571  -73.924942325642  
1       FELONY         40.536042        -74.153616  
2  MISDEMEANOR         40.537397        -74.239206  
3       FELONY          40.64594 

In [4]:
# ============================================
# CELL 3: See what crime types exist in NYC data
# ============================================

crime_raw = pd.read_csv(f'{base_path}/raw_data/nyc_crime_raw.csv')

print("Top 30 offense types in NYC data:")
print(crime_raw['ofns_desc'].value_counts().head(30))
print(f"\nTotal records: {len(crime_raw)}")
print(f"Total unique offense types: {crime_raw['ofns_desc'].nunique()}")

Top 30 offense types in NYC data:
ofns_desc
PETIT LARCENY                     504147
HARRASSMENT 2                     403680
ASSAULT 3 & RELATED OFFENSES      305181
CRIMINAL MISCHIEF & RELATED OF    287853
GRAND LARCENY                     254586
FELONY ASSAULT                    123579
OFF. AGNST PUB ORD SENSBLTY &     120775
DANGEROUS DRUGS                   105093
ROBBERY                            86062
MISCELLANEOUS PENAL LAW            80848
BURGLARY                           78617
DANGEROUS WEAPONS                  48750
OFFENSES AGAINST PUBLIC ADMINI     46927
GRAND LARCENY OF MOTOR VEHICLE     39215
VEHICLE AND TRAFFIC LAWS           39123
SEX CRIMES                         38148
INTOXICATED & IMPAIRED DRIVING     29264
FORGERY                            28875
THEFT-FRAUD                        24310
CRIMINAL TRESPASS                  20258
FRAUDS                             15980
POSSESSION OF STOLEN PROPERTY       9926
UNAUTHORIZED USE OF A VEHICLE       9841
RAPE         

In [5]:
# ============================================
# CELL 4: Filter to our 4 crime types
# ============================================

# Mapping NYC offense descriptions to our 4 categories
crime_type_mapping = {
    # ASSAULT
    'FELONY ASSAULT': 'ASSAULT',
    'ASSAULT 3 & RELATED OFFENSES': 'ASSAULT',

    # ROBBERY
    'ROBBERY': 'ROBBERY',

    # CRIMINAL DAMAGE (NYC calls it "Criminal Mischief")
    'CRIMINAL MISCHIEF & RELATED OF': 'CRIMINAL_DAMAGE',

    # THEFT (NYC uses Larceny)
    'PETIT LARCENY': 'THEFT',
    'GRAND LARCENY': 'THEFT',
    'GRAND LARCENY OF MOTOR VEHICLE': 'THEFT',
}

# Apply mapping
crime_raw['crime_type'] = crime_raw['ofns_desc'].map(crime_type_mapping)

# Keep only the 4 crime types (drop everything else)
crime_filtered = crime_raw.dropna(subset=['crime_type']).copy()

print(f"Records after filtering to 4 crime types: {len(crime_filtered)}")
print(f"\nBreakdown:")
print(crime_filtered['crime_type'].value_counts())

Records after filtering to 4 crime types: 1600623

Breakdown:
crime_type
THEFT              797948
ASSAULT            428760
CRIMINAL_DAMAGE    287853
ROBBERY             86062
Name: count, dtype: int64


In [6]:
# ============================================
# CELL 5: Clean dates, precincts, coordinates
# ============================================

# Parse dates
crime_filtered['date'] = pd.to_datetime(crime_filtered['cmplnt_fr_dt'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['date'])

# Extract just the date (no time)
crime_filtered['date'] = crime_filtered['date'].dt.date
crime_filtered['date'] = pd.to_datetime(crime_filtered['date'])

# Clean precinct column
crime_filtered['precinct'] = pd.to_numeric(crime_filtered['addr_pct_cd'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['precinct'])
crime_filtered['precinct'] = crime_filtered['precinct'].astype(int)

# Remove invalid precincts (NYC has precincts 1-123, but active ones are ~1-77)
crime_filtered = crime_filtered[(crime_filtered['precinct'] >= 1) & (crime_filtered['precinct'] <= 77)]

# Clean coordinates
crime_filtered['latitude'] = pd.to_numeric(crime_filtered['latitude'], errors='coerce')
crime_filtered['longitude'] = pd.to_numeric(crime_filtered['longitude'], errors='coerce')
crime_filtered = crime_filtered.dropna(subset=['latitude', 'longitude'])

# Filter date range: 2015-01-01 to 2020-03-10 (matching parent paper's ~5 year span)
crime_filtered = crime_filtered[
    (crime_filtered['date'] >= '2015-01-01') &
    (crime_filtered['date'] <= '2020-03-10')
]

print(f"✅ Clean dataset: {len(crime_filtered)} records")
print(f"Date range: {crime_filtered['date'].min()} to {crime_filtered['date'].max()}")
print(f"Precincts: {sorted(crime_filtered['precinct'].unique())}")
print(f"Number of precincts: {crime_filtered['precinct'].nunique()}")

✅ Clean dataset: 938414 records
Date range: 2015-01-01 00:00:00 to 2020-03-10 00:00:00
Precincts: [np.int64(1), np.int64(5), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(13), np.int64(14), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(28), np.int64(30), np.int64(32), np.int64(33), np.int64(34), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(52), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(75), np.int64(76), np.int64(77)]
Number of precincts: 49


In [7]:
# ============================================
# CELL 6: Create daily crime counts per precinct per type
# This is the core dataset for modeling
# ============================================
from itertools import product

# Pivot: for each date + precinct, count each crime type
daily_counts = crime_filtered.groupby(['date', 'precinct', 'crime_type']).size().reset_index(name='count')

# Pivot wider: one column per crime type
daily_pivot = daily_counts.pivot_table(
    index=['date', 'precinct'],
    columns='crime_type',
    values='count',
    fill_value=0
).reset_index()

# Flatten column names
daily_pivot.columns = ['date', 'precinct', 'assault', 'criminal_damage', 'robbery', 'theft']

# Make sure ALL dates × ALL precincts exist (fill missing with 0)
all_dates = pd.date_range(start='2015-01-01', end='2020-03-10', freq='D')
all_precincts = sorted(crime_filtered['precinct'].unique())

# Create complete index
complete_index = pd.DataFrame(
    list(product(all_dates, all_precincts)),
    columns=['date', 'precinct']
)

# Merge — fills missing combos with 0
daily_complete = complete_index.merge(daily_pivot, on=['date', 'precinct'], how='left')
daily_complete = daily_complete.fillna(0)

# Convert counts to int
for col in ['assault', 'criminal_damage', 'robbery', 'theft']:
    daily_complete[col] = daily_complete[col].astype(int)

print(f"✅ Complete daily dataset: {len(daily_complete)} rows")
print(f"   ({len(all_dates)} days × {len(all_precincts)} precincts)")
print(daily_complete.head(10))

# Save
daily_complete.to_csv(f'{base_path}/processed_data/daily_crime_counts.csv', index=False)
print(f"\n✅ Saved to: processed_data/daily_crime_counts.csv")

✅ Complete daily dataset: 92904 rows
   (1896 days × 49 precincts)
        date  precinct  assault  criminal_damage  robbery  theft
0 2015-01-01         1        0                5        0      3
1 2015-01-01         5        2                1        0      3
2 2015-01-01         6        5               17        0     12
3 2015-01-01         7        9                1        1      3
4 2015-01-01         9        6               13        0     13
5 2015-01-01        10        2                1        1      8
6 2015-01-01        13        5                2        0     13
7 2015-01-01        14        4                1        1     18
8 2015-01-01        17        1                1        0      5
9 2015-01-01        18        9                0        0     13

✅ Saved to: processed_data/daily_crime_counts.csv


In [8]:
# ============================================
# CELL 7 (FIXED): Download NYC weather data
# ============================================
import openmeteo_requests
import requests_cache
from retry_requests import retry
import numpy as np

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
weather_frames = []

for year in range(2015, 2021):
    end_date = f"{year}-12-31" if year < 2020 else "2020-03-10"
    print(f"  Downloading weather {year}...")

    params = {
        "latitude": 40.7831,
        "longitude": -73.9712,
        "start_date": f"{year}-01-01",
        "end_date": end_date,
        "daily": ["temperature_2m_mean", "relative_humidity_2m_mean",
                   "wind_speed_10m_max", "precipitation_sum"],
        "timezone": "America/New_York"
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    daily = response.Daily()

    # Fix: create date range manually using start, end, and daily frequency
    start_date = pd.to_datetime(daily.Time(), unit="s", utc=True)
    end_date_ts = pd.to_datetime(daily.TimeEnd(), unit="s", utc=True)
    dates = pd.date_range(start=start_date, end=end_date_ts, freq="D", inclusive="left")

    weather_df = pd.DataFrame({
        "date": dates,
        "temperature": daily.Variables(0).ValuesAsNumpy()[:len(dates)],
        "humidity": daily.Variables(1).ValuesAsNumpy()[:len(dates)],
        "wind_speed": daily.Variables(2).ValuesAsNumpy()[:len(dates)],
        "precipitation": daily.Variables(3).ValuesAsNumpy()[:len(dates)],
    })
    weather_frames.append(weather_df)

weather = pd.concat(weather_frames, ignore_index=True)
weather['date'] = weather['date'].dt.tz_localize(None)
weather['date'] = pd.to_datetime(weather['date'].dt.date)

print(f"\n✅ Weather data: {len(weather)} days")
print(weather.head())

weather.to_csv(f'{base_path}/raw_data/nyc_weather.csv', index=False)
print(f"✅ Saved to: raw_data/nyc_weather.csv")


✅ Weather data: 1896 days
        date  temperature   humidity  wind_speed  precipitation
0 2015-01-01    -1.787000  43.443012   21.267441            0.0
1 2015-01-02     0.598417  58.507015   19.881649            0.0
2 2015-01-03     0.463000  82.078606   15.827721           16.5
3 2015-01-04     8.765083  96.569328   22.346113            7.2
4 2015-01-05     0.802584  50.426418   26.756500            0.0
✅ Saved to: raw_data/nyc_weather.csv


In [9]:
# ============================================
# CELL 8: Create holiday and temporal features
# ============================================

# US Federal Holidays 2015-2020
us_holidays = [
    # 2015
    '2015-01-01', '2015-01-19', '2015-02-16', '2015-05-25', '2015-07-03',
    '2015-09-07', '2015-10-12', '2015-11-11', '2015-11-26', '2015-12-25',
    # 2016
    '2016-01-01', '2016-01-18', '2016-02-15', '2016-05-30', '2016-07-04',
    '2016-09-05', '2016-10-10', '2016-11-11', '2016-11-24', '2016-12-26',
    # 2017
    '2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29', '2017-07-04',
    '2017-09-04', '2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25',
    # 2018
    '2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28', '2018-07-04',
    '2018-09-03', '2018-10-08', '2018-11-12', '2018-11-22', '2018-12-25',
    # 2019
    '2019-01-01', '2019-01-21', '2019-02-18', '2019-05-27', '2019-07-04',
    '2019-09-02', '2019-10-14', '2019-11-11', '2019-11-28', '2019-12-25',
    # 2020
    '2020-01-01', '2020-01-20', '2020-02-17',
]
us_holidays = pd.to_datetime(us_holidays)

# Create temporal features for each date
all_dates = pd.date_range(start='2015-01-01', end='2020-03-10', freq='D')
temporal = pd.DataFrame({'date': all_dates})

# Weekend feature: 1 = weekend, 0 = weekday
temporal['weekend'] = (temporal['date'].dt.dayofweek >= 5).astype(int)

# Holiday feature: 1 = holiday, 0 = not
temporal['holiday'] = temporal['date'].isin(us_holidays).astype(int)

# Day of week and Month
temporal['day_of_week'] = temporal['date'].dt.dayofweek
temporal['month'] = temporal['date'].dt.month

print(f"✅ Temporal features: {len(temporal)} days")
print(f"   Weekends: {temporal['weekend'].sum()}")
print(f"   Holidays: {temporal['holiday'].sum()}")
print(temporal.head(10))

temporal.to_csv(f'{base_path}/processed_data/temporal_features.csv', index=False)
print(f"\n✅ Saved to: processed_data/temporal_features.csv")

✅ Temporal features: 1896 days
   Weekends: 542
   Holidays: 53
        date  weekend  holiday  day_of_week  month
0 2015-01-01        0        1            3      1
1 2015-01-02        0        0            4      1
2 2015-01-03        1        0            5      1
3 2015-01-04        1        0            6      1
4 2015-01-05        0        0            0      1
5 2015-01-06        0        0            1      1
6 2015-01-07        0        0            2      1
7 2015-01-08        0        0            3      1
8 2015-01-09        0        0            4      1
9 2015-01-10        1        0            5      1

✅ Saved to: processed_data/temporal_features.csv


In [10]:
# ============================================
# CELL 9: Compute derived weather features
# Formulas from Appendix A of parent paper
# ============================================
import numpy as np

weather = pd.read_csv(f'{base_path}/raw_data/nyc_weather.csv')
weather['date'] = pd.to_datetime(weather['date'])

# Fill any missing values
weather = weather.fillna(method='ffill').fillna(method='bfill')

T = weather['temperature'].values    # Air temperature (°C)
RH = weather['humidity'].values      # Relative humidity (%)
V = weather['wind_speed'].values     # Wind speed (m/s)

# Water vapor pressure (Equation A3)
e = (RH / 100) * 6.105 * np.exp((17.2 * T) / (237.7 + T))

# Apparent Temperature (Equation A2)
AT = 1.07 * T + 0.2 * e - 0.65 * V - 2.7

# Thermodynamic wet-bulb temperature (Equation A4)
a, b, c, d_coef, e_coef, f_coef = 0.151977, 8.313659, 1.676311, 0.00391838, 0.023101, 4.68035
Tw = (AT * np.arctan(a * np.sqrt(RH + b))
      + np.arctan(AT + RH)
      - np.arctan(RH - c)
      + d_coef * (RH ** 1.5) * np.arctan(e_coef * RH)
      - f_coef)

# Discomfort Index (Equation A1)
DI = 0.5 * Tw + 0.5 * AT

# Add to weather dataframe
weather['water_vapor_pressure'] = e
weather['apparent_temp'] = AT
weather['wet_bulb_temp'] = Tw
weather['discomfort_index'] = DI

print("✅ Weather features with derived variables:")
print(weather.head())
print(f"\nColumns: {list(weather.columns)}")

weather.to_csv(f'{base_path}/processed_data/weather_features.csv', index=False)
print(f"\n✅ Saved to: processed_data/weather_features.csv")

/tmp/ipykernel_4220/3108175570.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  weather = weather.fillna(method='ffill').fillna(method='bfill')


✅ Weather features with derived variables:
        date  temperature   humidity  wind_speed  precipitation  \
0 2015-01-01    -1.787000  43.443012   21.267440            0.0   
1 2015-01-02     0.598417  58.507015   19.881649            0.0   
2 2015-01-03     0.463000  82.078606   15.827721           16.5   
3 2015-01-04     8.765083  96.569330   22.346113            7.2   
4 2015-01-05     0.802584  50.426420   26.756500            0.0   

   water_vapor_pressure  apparent_temp  wet_bulb_temp  discomfort_index  
0              2.328213     -17.970284     -18.727137        -18.348710  
1              3.729512     -14.236864     -15.762054        -14.999459  
2              5.181284     -11.456352     -12.579311        -12.017832  
3             10.868687      -5.672597      -6.077811         -5.875204  
4              3.261975     -18.580566     -19.487329        -19.033947  

Columns: ['date', 'temperature', 'humidity', 'wind_speed', 'precipitation', 'water_vapor_pressure', 'apparent

In [11]:
# ============================================
# CELL 10: Merge crime + weather + temporal into final dataset
# ============================================

# Load all processed data
daily_crime = pd.read_csv(f'{base_path}/processed_data/daily_crime_counts.csv')
weather_feat = pd.read_csv(f'{base_path}/processed_data/weather_features.csv')
temporal_feat = pd.read_csv(f'{base_path}/processed_data/temporal_features.csv')

# Convert dates
daily_crime['date'] = pd.to_datetime(daily_crime['date'])
weather_feat['date'] = pd.to_datetime(weather_feat['date'])
temporal_feat['date'] = pd.to_datetime(temporal_feat['date'])

# Merge crime with temporal features
merged = daily_crime.merge(temporal_feat, on='date', how='left')

# Merge with weather (weather is per date, not per precinct)
merged = merged.merge(weather_feat, on='date', how='left')

# Sort by precinct and date
merged = merged.sort_values(['precinct', 'date']).reset_index(drop=True)

# Create rolling average features (per precinct, per crime type)
for crime_col in ['assault', 'criminal_damage', 'robbery', 'theft']:
    # 30-day rolling average
    merged[f'{crime_col}_weekday_avg'] = merged.groupby('precinct')[crime_col].transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
    # Month average
    merged[f'{crime_col}_month_avg'] = merged.groupby('precinct')[crime_col].transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
    # Previous day count
    merged[f'{crime_col}_prev'] = merged.groupby('precinct')[crime_col].shift(1).fillna(0)

print(f"✅ Final merged dataset: {merged.shape}")
print(f"   Columns: {list(merged.columns)}")
print(merged.head())

merged.to_csv(f'{base_path}/processed_data/final_dataset.csv', index=False)
print(f"\n✅ Saved to: processed_data/final_dataset.csv")

✅ Final merged dataset: (92904, 30)
   Columns: ['date', 'precinct', 'assault', 'criminal_damage', 'robbery', 'theft', 'weekend', 'holiday', 'day_of_week', 'month', 'temperature', 'humidity', 'wind_speed', 'precipitation', 'water_vapor_pressure', 'apparent_temp', 'wet_bulb_temp', 'discomfort_index', 'assault_weekday_avg', 'assault_month_avg', 'assault_prev', 'criminal_damage_weekday_avg', 'criminal_damage_month_avg', 'criminal_damage_prev', 'robbery_weekday_avg', 'robbery_month_avg', 'robbery_prev', 'theft_weekday_avg', 'theft_month_avg', 'theft_prev']
        date  precinct  assault  criminal_damage  robbery  theft  weekend  \
0 2015-01-01         1        0                5        0      3        0   
1 2015-01-02         1        2                0        0      6        0   
2 2015-01-03         1        0                0        0      5        1   
3 2015-01-04         1        2                0        0      9        1   
4 2015-01-05         1        0                0        

In [12]:
# ============================================
# CELL 11 (FIXED): Build adjacency matrix for NYC precincts
# Using a different data source
# ============================================
import geopandas as gpd
import numpy as np
import requests
import json

print("⏳ Downloading NYC precinct boundaries...")

# Try the direct GeoJSON URL from NYC Open Data
url = "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Police_Precincts/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
response = requests.get(url)

# Check if we got valid JSON
try:
    data = response.json()
    print(f"  Downloaded {len(data.get('features', []))} features")

    # Save and load
    filepath = f'{base_path}/raw_data/nyc_precincts.geojson'
    with open(filepath, 'w') as f:
        json.dump(data, f)

    precincts_gdf = gpd.read_file(filepath)
    print(f"  ✅ Loaded from ArcGIS")

except:
    # Fallback: build adjacency from crime data coordinates
    print("  ⚠️ Could not download shapefile. Building adjacency from crime coordinates...")

    crime_raw = pd.read_csv(f'{base_path}/raw_data/nyc_crime_raw.csv')
    crime_raw['precinct'] = pd.to_numeric(crime_raw['addr_pct_cd'], errors='coerce')
    crime_raw['latitude'] = pd.to_numeric(crime_raw['latitude'], errors='coerce')
    crime_raw['longitude'] = pd.to_numeric(crime_raw['longitude'], errors='coerce')
    crime_raw = crime_raw.dropna(subset=['precinct', 'latitude', 'longitude'])
    crime_raw['precinct'] = crime_raw['precinct'].astype(int)

    # Get centroid of each precinct
    centroids = crime_raw.groupby('precinct').agg(
        lat=('latitude', 'mean'),
        lon=('longitude', 'mean')
    ).reset_index()

    crime_precincts = sorted(pd.read_csv(f'{base_path}/processed_data/daily_crime_counts.csv')['precinct'].unique())
    centroids = centroids[centroids['precinct'].isin(crime_precincts)].sort_values('precinct').reset_index(drop=True)

    n = len(centroids)
    precinct_ids = centroids['precinct'].values
    adj_matrix = np.zeros((n, n), dtype=int)

    # Connect precincts that are within ~3km of each other
    from math import radians, cos, sin, asin, sqrt

    def haversine(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        return 2 * 6371 * asin(sqrt(a))  # km

    THRESHOLD_KM = 3.0

    for i in range(n):
        for j in range(i+1, n):
            dist = haversine(centroids.iloc[i]['lat'], centroids.iloc[i]['lon'],
                           centroids.iloc[j]['lat'], centroids.iloc[j]['lon'])
            if dist <= THRESHOLD_KM:
                adj_matrix[i, j] = 1
                adj_matrix[j, i] = 1

    np.fill_diagonal(adj_matrix, 1)

    print(f"\n✅ Adjacency matrix shape: {adj_matrix.shape}")
    print(f"   Total edges (excluding self-loops): {(adj_matrix.sum() - n) // 2}")
    print(f"   Avg neighbors per precinct: {(adj_matrix.sum() - n) / n:.1f}")

    np.save(f'{base_path}/processed_data/adjacency_matrix.npy', adj_matrix)
    np.save(f'{base_path}/processed_data/precinct_ids.npy', precinct_ids)
    print("✅ Saved adjacency matrix and precinct IDs")

# If shapefile loaded successfully, build adjacency from geometry
if 'precincts_gdf' in dir() and precincts_gdf is not None and len(precincts_gdf) > 0:
    print(f"\nColumns: {list(precincts_gdf.columns)}")

    # Find precinct column
    precinct_col = None
    for col in precincts_gdf.columns:
        if 'precinct' in col.lower():
            precinct_col = col
            break
    if precinct_col is None:
        for col in precincts_gdf.columns:
            try:
                vals = pd.to_numeric(precincts_gdf[col], errors='coerce').dropna()
                if len(vals) > 0 and vals.min() >= 1 and vals.max() <= 130:
                    precinct_col = col
                    break
            except:
                pass

    if precinct_col:
        precincts_gdf['precinct_id'] = pd.to_numeric(precincts_gdf[precinct_col], errors='coerce').astype(int)

        crime_precincts = sorted(pd.read_csv(f'{base_path}/processed_data/daily_crime_counts.csv')['precinct'].unique())
        precincts_gdf = precincts_gdf[precincts_gdf['precinct_id'].isin(crime_precincts)].copy()
        precincts_gdf = precincts_gdf.sort_values('precinct_id').reset_index(drop=True)

        n = len(precincts_gdf)
        precinct_ids = precincts_gdf['precinct_id'].values
        adj_matrix = np.zeros((n, n), dtype=int)

        print(f"⏳ Building adjacency matrix ({n}×{n})...")

        for i in range(n):
            for j in range(i+1, n):
                geom_i = precincts_gdf.iloc[i].geometry
                geom_j = precincts_gdf.iloc[j].geometry
                if geom_i.buffer(0.001).intersects(geom_j):
                    adj_matrix[i, j] = 1
                    adj_matrix[j, i] = 1

        np.fill_diagonal(adj_matrix, 1)

        print(f"\n✅ Adjacency matrix shape: {adj_matrix.shape}")
        print(f"   Total edges (excluding self-loops): {(adj_matrix.sum() - n) // 2}")
        print(f"   Avg neighbors per precinct: {(adj_matrix.sum() - n) / n:.1f}")

        np.save(f'{base_path}/processed_data/adjacency_matrix.npy', adj_matrix)
        np.save(f'{base_path}/processed_data/precinct_ids.npy', precinct_ids)
        print("✅ Saved adjacency matrix and precinct IDs")

⏳ Downloading NYC precinct boundaries...
  Downloaded 77 features
  ✅ Loaded from ArcGIS

Columns: ['OBJECTID', 'Precinct', 'Shape__Area', 'Shape__Length', 'geometry']
⏳ Building adjacency matrix (49×49)...

✅ Adjacency matrix shape: (49, 49)
   Total edges (excluding self-loops): 99
   Avg neighbors per precinct: 4.0
✅ Saved adjacency matrix and precinct IDs


In [13]:
# ============================================
# CELL 12: Split into training and test sets
# Train: 2015-01-01 to 2019-12-31
# Test: 2020-01-01 to 2020-01-07 (7 days, matching parent paper)
# ============================================

merged = pd.read_csv(f'{base_path}/processed_data/final_dataset.csv')
merged['date'] = pd.to_datetime(merged['date'])

train_end = '2019-12-31'
test_start = '2020-01-01'
test_end = '2020-01-07'

train_data = merged[merged['date'] <= train_end].copy()
test_data = merged[(merged['date'] >= test_start) & (merged['date'] <= test_end)].copy()

print(f"✅ Train set: {len(train_data)} rows ({train_data['date'].min().date()} to {train_data['date'].max().date()})")
print(f"✅ Test set:  {len(test_data)} rows ({test_data['date'].min().date()} to {test_data['date'].max().date()})")
print(f"\n   Train days: {train_data['date'].nunique()}")
print(f"   Test days:  {test_data['date'].nunique()}")
print(f"   Precincts:  {train_data['precinct'].nunique()}")

train_data.to_csv(f'{base_path}/processed_data/train_data.csv', index=False)
test_data.to_csv(f'{base_path}/processed_data/test_data.csv', index=False)
print("\n✅ Saved train and test data")

✅ Train set: 89474 rows (2015-01-01 to 2019-12-31)
✅ Test set:  343 rows (2020-01-01 to 2020-01-07)

   Train days: 1826
   Test days:  7
   Precincts:  49

✅ Saved train and test data
